# Brand Logo Detection (YOLOv8)

Prepare LogoDet-3K (or compatible) data, train the YOLOv8 logo detector, and verify inference with the brand API model.


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
for cand in [repo_root, *repo_root.parents]:
    if (cand / 'app').exists():
        repo_root = cand
        break
sys.path.insert(0, str(repo_root))
print('Repo root:', repo_root)


## Data requirements

Place LogoDet-3K (or a compatible dataset) under data/raw/brand/LogoDet-3K or data/raw/brand/logodet3k. The preparation script will convert it into data/processed/brand_yolo.


## Links to code


- Data prep: `scripts/prepare_brand_data.py`


- Training: `src/train/train_brand_logo_detector.py`


- Inference module: `src/vision/brand/recognizer.py`


- API endpoint: `POST /api/vision/brand/predict`


In [ ]:
# Update paths if your data lives elsewhere.



In [ ]:
raw_dir = repo_root / 'data' / 'raw' / 'brand'
print('Raw brand dir:', raw_dir, 'exists=', raw_dir.exists())


## Dataset inventory


In [ ]:
from pathlib import Path

candidates = [
    repo_root / 'data' / 'raw' / 'brand' / 'LogoDet-3K',
    repo_root / 'data' / 'raw' / 'brand' / 'logodet3k',
    repo_root / 'data' / 'raw' / 'brand',
]
dataset_root = next((c for c in candidates if c.exists()), None)
print('Dataset root:', dataset_root)

if dataset_root:
    img_exts = {'.jpg', '.jpeg', '.png', '.webp'}
    img_count = sum(1 for p in dataset_root.rglob('*') if p.suffix.lower() in img_exts)
    txt_count = sum(1 for p in dataset_root.rglob('*.txt'))
    xml_count = sum(1 for p in dataset_root.rglob('*.xml'))
    json_count = sum(1 for p in dataset_root.rglob('*.json'))
    print('Images:', img_count, 'labels txt:', txt_count, 'xml:', xml_count, 'json:', json_count)
else:
    print('No raw brand dataset found in the default locations.')


## Processed YOLO dataset preview


In [ ]:
processed_root = repo_root / 'data' / 'processed' / 'brand_yolo'
yaml_path = processed_root / 'brands.yaml'
print('Processed root exists:', processed_root.exists())
if yaml_path.exists():
    try:
        import yaml
    except Exception as exc:
        print('PyYAML not available:', exc)
    else:
        cfg = yaml.safe_load(yaml_path.read_text(encoding='utf-8')) or {}
        print('YOLO config:', cfg)
        names = cfg.get('names') or []
        print('Classes:', names)

        def _count_images(entry):
            from pathlib import Path
            if entry is None:
                return 0
            if isinstance(entry, (list, tuple)):
                return sum(_count_images(v) for v in entry)
            p = Path(str(entry))
            if p.is_file() and p.suffix.lower() == '.txt':
                return len([l for l in p.read_text(encoding='utf-8').splitlines() if l.strip()])
            if p.exists() and p.is_dir():
                return sum(1 for q in p.rglob('*') if q.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'})
            return 0

        base = Path(cfg.get('path') or yaml_path.parent)
        train_entry = cfg.get('train')
        val_entry = cfg.get('val') or cfg.get('test')
        train_count = _count_images(base / train_entry) if train_entry else 0
        val_count = _count_images(base / val_entry) if val_entry else 0
        print('Train images:', train_count)
        print('Val images:', val_count)
else:
    print('brands.yaml not found yet. Run scripts/prepare_brand_data.py first.')


## Sample labeled image (YOLO labels)


In [ ]:
from PIL import Image, ImageDraw

processed_root = repo_root / 'data' / 'processed' / 'brand_yolo'
img_dir = processed_root / 'images' / 'train'
label_dir = processed_root / 'labels' / 'train'

sample = None
if img_dir.exists():
    for p in img_dir.rglob('*'):
        if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}:
            sample = p
            break

if sample is None:
    print('No processed training image found under', img_dir)
else:
    rel = sample.relative_to(img_dir)
    label_path = (label_dir / rel).with_suffix('.txt')
    img = Image.open(sample).convert('RGB')
    draw = ImageDraw.Draw(img)
    if label_path.exists():
        w, h = img.size
        for line in label_path.read_text(encoding='utf-8').splitlines():
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            _, cx, cy, bw, bh = map(float, parts[:5])
            x1 = (cx - bw / 2.0) * w
            y1 = (cy - bh / 2.0) * h
            x2 = (cx + bw / 2.0) * w
            y2 = (cy + bh / 2.0) * h
            draw.rectangle([x1, y1, x2, y2], outline='lime', width=2)
        print('Label file:', label_path)
    else:
        print('Label file missing for', sample)
    img.show()


## Prepare dataset into YOLO format


In [ ]:
# !python scripts/prepare_brand_data.py


## Train YOLOv8 detector


In [ ]:
# Quick test (adjust epochs/imgsz/batch as needed).
# You can set BRAND_TRAIN_MAX_IMAGES to limit images for a smoke test.

# Example:
# BRAND_YOLO_MODEL=yolov8n.pt BRAND_EPOCHS=10 BRAND_IMGSZ=320 BRAND_BATCH=16 \
# BRAND_DEVICE=auto BRAND_TRAIN_MAX_IMAGES=5000 BRAND_VAL=true \
# python -m src.train.train_brand_logo_detector


## Run a local inference sample


In [ ]:
from pathlib import Path
from src.vision.brand.recognizer import predict_image_bytes

sample = None
for cand in (repo_root / 'data' / 'processed' / 'brand_yolo').rglob('*.jpg'):
    sample = cand
    break

if sample is None:
    print('No sample image found under data/processed/brand_yolo')
else:
    detections = predict_image_bytes(sample.read_bytes(), conf=0.25)
    print('Detections:', detections[:5])


## Visuals and metrics


In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from src.vision.brand.recognizer import predict_image_bytes

sample = None
candidate_dir = repo_root / 'data' / 'processed' / 'brand_yolo'
if candidate_dir.exists():
    for p in candidate_dir.rglob('*.jpg'):
        sample = p
        break

if sample is None:
    print('No sample image found under', candidate_dir)
else:
    img = Image.open(sample).convert('RGB')
    detections = predict_image_bytes(sample.read_bytes(), conf=0.25)
    draw = ImageDraw.Draw(img)
    for det in detections:
        bbox = det.get('bbox', [])
        if len(bbox) == 4:
            draw.rectangle(bbox, outline='red', width=2)
            label = '{} {:.2f}'.format(det.get('brand'), det.get('confidence', 0.0))
            draw.text((bbox[0], bbox[1]), label, fill='red')
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Brand detections')
    plt.show()
    if detections:
        scores = [d.get('confidence', 0.0) for d in detections]
        plt.figure(figsize=(4, 3))
        plt.hist(scores, bins=10)
        plt.title('Detection confidence')
        plt.xlabel('confidence')
        plt.ylabel('count')
        plt.show()
